In [3]:
from tqdm import tqdm
import os
import glob
import pandas as pd

In [4]:
# funzione che legge i file .book e restituisce una lista di dizionari (top N=10 personaggi)

def read_booknlp(path_book):
    with open(path_book, "r", encoding="utf-8") as file:
        lines = file.readlines()
        # eval() transforme une chaîne de caractères représentant un dictionnaire en objet Python
        dicts = [eval(line.strip()) for line in lines if line.strip()]
    return dicts[:5] # /!\ On ne prends que les 5 premiers personnages pour un roman

In [5]:
# Define the folder where your .book files are located (adjust as needed)
books_folder = "/Users/deniseatzori/Library/Mobile Documents/com~apple~CloudDocs/ENC-PSL/Memoire/Github_tesi/Corpus_propp"

book_files = glob.glob(os.path.join(books_folder, "*.book"))

In [6]:
def get_character_name(char_dict):
    """
    Retrieve a character name from the 'mentions' field.
    If no proper mention is available, returns a fallback name using the character id.
    """
    if "mentions" in char_dict:
        proper = char_dict["mentions"].get("proper", [])
        if len(proper) > 0:
            # Use the first proper mention (which is usually the most frequent)
            return proper[0]["n"].lower()
    return f"char{char_dict.get('id', 'unknown')}"

In [7]:
# ------------------------------
# 2) Build df_chapitres from All .book Files in a Folder
# ------------------------------

data_rows = []

# Loop over every .book file
for file_path in tqdm(book_files):
    base_name = os.path.basename(file_path).replace(".book", "")
    # Read the first n characters from the file
    book_data = read_booknlp(file_path)
    
    # Process each character dictionary
    for char_dict in book_data:
        char_id = char_dict.get("id", None)
        char_name = get_character_name(char_dict)

        # add the gender prediction
        if book_data[char_id]['gender'].get("argmax") == "Female":
            gender = 1
        else:
            gender = 0

        
        # Prepare a row dictionary with metadata.
        row_data = {
            "title" : f"{base_name}.book --- {char_name}", # nome del libro + nome del pg
            "text_id": base_name,        # file (text) id
            "char_id": char_id,          # character id from BookNLP
            "char_name": char_name,      # derived character name
            "type" : "",
            "Gender":""
        }
        data_rows.append(row_data)

# Build the output dataframe from all rows
df_chapitres = pd.DataFrame(data_rows)
print("df_chapitres shape:", df_chapitres.shape)

100%|██████████| 123/123 [00:19<00:00,  6.38it/s]

df_chapitres shape: (615, 6)


In [8]:
df_chapitres.to_csv("total_characters.csv", header=True, index=False)